# Causal Discovery and Causal Feature Selection for Robust Prediction

This tutorial walks through the key ideas and algorithms at the intersection of causal inference and robust machine learning.
The tutorial is structured as follows:

1. **Pairwise causal discovery** — can we tell which variable causes which, from observational data alone?
2. **Multivariate causal discovery** — recovering the full causal graph over many variables
3. **Multivariate causal feature selection** — finding the Markov Blanket of a target variable
4. **CFS for robust prediction** — why causal parents are more stable predictors than correlated features
5. **Invariant Causal Predictors** — a principled method to find invariant features across environments

In [ ]:
import os
if not os.path.exists('zh03-causal-discovery-robust-predictions'):
    !git clone https://github.com/WinterSchool2026/zh03-causal-discovery-robust-predictions.git
import sys
sys.path.insert(0, 'zh03-causal-discovery-robust-predictions')

In [ ]:
pip install causal-learn

In [ ]:
# Standard library
import time

# Core libraries
import numpy as np
import pandas as pd

# SciPy / stats
from scipy.stats import pearsonr
from numpy.linalg import lstsq

# causal-learn
from causallearn.search.ConstraintBased.PC import pc
from causallearn.search.ScoreBased.GES import ges
from causallearn.utils.cit import fisherz
from src.ci_test import *
from src.ci_test import ci_table


# Local project modules
from src.generate_scm import *

# Reproducibility
np.random.seed(2)


---
## Section 1.5 — Markov Equivalence Classes

Consider three variables $X0, X1, X2$ and the following three DAGs:

```
(1)  $X_0$ → $X_1$ → $X_2$        (chain)
(2)  $X_0$ ← $X_1$ → $X_2$        (fork / common cause)
(3)  $X_0$ → $X_1$ ← $X_2$        (collider / v-structure)
```

In DAGs (1) and (2), $X_0$ and $X_2$ are **marginally dependent** but **conditionally independent given $X_1$** — written X0 ⊥ X2 | X1. In DAG (3), $X_0$ and $X_2$ are **marginally independent** but become **dependent when conditioning on $X_1$** (the collider).

This distinction is testable from data. But crucially, **(1) and (2) are indistinguishable from observational data alone**: they encode exactly the same conditional independencies. No statistical test — without further assumptions — can tell them apart.

### 1.5.2 Markov Equivalence Classes (MEC)

Two DAGs are **Markov equivalent** if they have the same skeleton (undirected edges) and the same **v-structures** (patterns $X → Z ← Y$ where $X$ and $Y$ are not adjacent). 

The set of all Markov-equivalent DAGs forms the **Markov Equivalence Class (MEC)**. Constraint-based algorithms can recover only the MEC. Their output is a **CPDAG** (Completed Partially Directed Acyclic Graph): edges that point the same way across all DAGs in the MEC appear as arrows; edges that could go either way appear as undirected lines.

### 1.5.3 What PC orients and what it leaves undirected

PC proceeds in two phases:

1. **Skeleton recovery**: remove edges wherever a conditional independence is found.
2. **Orientation**: orient v-structures (forced by the data), then propagate via Meek rules without creating new v-structures or cycles.

Edges that cannot be oriented by these rules remain undirected.

In [ ]:

# ----------------------------------------------------------------
# Build three 3-variable SCMs manually via adjacency matrices
# Variables: X0, X1, X2  (d=2, so Y is X2 in the SCM framework)
# ----------------------------------------------------------------
d = 2  # 2 non-target vars + target Y => 3 nodes total

def make_scm_from_adj(A_matrix, seed=13):
    np.random.seed(seed)
    scm = SCMGenerator(d=d)
    scm.fit_from_adjacency(A_matrix, Y_idx=d,
                           noise_type='uniform', is_linear=True)
    return scm

col_names = ['X0', 'X1', 'Y']  # Y plays the role of X2

# (1) Chain:    X0 -> X1 -> Y
A_chain = np.array([[0, 1, 0],
                    [0, 0, 1],
                    [0, 0, 0]], dtype=float)

# (2) Fork:     X0 <- X1 -> Y   (X1 is the common cause)
A_fork  = np.array([[0, 0, 0],
                    [1, 0, 1],
                    [0, 0, 0]], dtype=float)

# (3) Collider: X0 -> X1 <- Y
A_coll  = np.array([[0, 1, 0],
                    [0, 0, 0],
                    [0, 1, 0]], dtype=float)

scm_chain = make_scm_from_adj(A_chain)
scm_fork  = make_scm_from_adj(A_fork)
scm_coll  = make_scm_from_adj(A_coll)

print('True DAGs:')
plot_graphs_from_adj([A_chain, A_fork, A_coll],
                     Y_idx_list=[d, d, d],
                     plot_titles=['(1) Chain  X0->X1->Y',
                                  '(2) Fork   X0<-X1->Y',
                                  '(3) Collider X0->X1<-Y'])


### 1.5.5 Step 1 — Enumerate all pairwise conditional independencies

Let's read the CI structure **directly from the data** using Fisher's z-test.
For each pair of variables $(X, Y)$ we test:

- **Marginal independence**: X ⊥ Y  (no conditioning set)
- **Conditional independence given the third variable**: X ⊥ Y | Z

Collecting all CI statements that hold at α = 0.05 gives us the **empirical CI oracle** that any constraint-based algorithm will implicitly rely on.


In [ ]:
n_samples = 2000
data_chain = scm_chain.sample(n_samples=n_samples)
ci_table(data_chain, data_chain.columns)


In [ ]:
data_fork  = scm_fork.sample(n_samples=n_samples)
ci_table(data_fork,  'Fork    (X0 ← X1 → Y)')


In [ ]:
data_coll  = scm_coll.sample(n_samples=n_samples)
ci_table(data_coll,  'Collider (X0 → X1 ← Y)')

### 1.5.6 Step 2 — Run PC and verify against the CI oracle

Now we let PC do the same work automatically. It will:
1. Test CI statements (same Fisher z-test, same α = 0.05)
2. Remove edges where independence holds → skeleton
3. Orient v-structures, then apply Meek rules → CPDAG

The result should be **exactly consistent** with the CI statements you identified above.


In [ ]:
def run_pc(data, col_names, label, alpha=0.05):
    X = data[col_names].values
    t0 = time.time()
    cg = pc(X, alpha=alpha, indep_test=fisherz)
    A = get_adjacency_pc(cg, col_names)
    print(f'{label}  |  PC (α={alpha}): {time.time()-t0:.2f}s')
    return A

print('Running PC on all three SCMs (n=2000, α=0.05)...')
pc_ch = run_pc(data_chain, col_names, 'Chain   ')
pc_fo = run_pc(data_fork,  col_names, 'Fork    ')
pc_co = run_pc(data_coll,  col_names, 'Collider')

print()
plot_graphs_from_adj(
    [A_chain, pc_ch.values],
    Y_idx_list=[d, d],
    plot_titles=['True: Chain', 'PC recovered'])

plot_graphs_from_adj(
    [A_fork, pc_fo.values],
    Y_idx_list=[d, d],
    plot_titles=['True: Fork', 'PC recovered'])

plot_graphs_from_adj(
    [A_coll, pc_co.values],
    Y_idx_list=[d, d],
    plot_titles=['True: Collider', 'PC recovered'])


### 1.5.7 Step 3 — How PC fails: sensitivity to sample size and α

PC's skeleton recovery depends on statistical tests. When samples are scarce or α is poorly chosen:
- **Too few samples** → tests lack power, true edges may survive (false negatives in removal = spurious edges kept) or be incorrectly removed.
- **α too large** → edges are removed too aggressively (over-pruning, missing true edges in skeleton).
- **α too small** → spurious edges survive (under-pruning, false connections).

We test this by varying both axes on the **Collider** SCM (where orientation matters most).


In [ ]:
sample_sizes = [50, 200, 500, 2000]
alphas        = [0.001, 0.01, 0.05, 0.20]

# True collider adjacency for comparison
A_true_coll = A_coll  # shape (3,3)

print(f'{"":18}', end='')
for a in alphas:
    print(f'  α={a:<5}', end='')
print()
print('-' * (18 + len(alphas) * 12))

for n in sample_sizes:
    np.random.seed(42)
    df = scm_coll.sample(n_samples=n)
    X_np = df[col_names].values
    row_str = f'n={n:<6} (collider)  '
    for a in alphas:
        cg = pc(X_np, alpha=a, indep_test=fisherz)
        A_rec = get_adjacency_pc(cg, col_names).values
        # Count TP (true edges recovered), FP (spurious edges), FN (missed edges)
        true_edges = int(A_true_coll.sum())
        tp = int(((A_rec > 0) & (A_true_coll > 0)).sum())
        fp = int(((A_rec > 0) & (A_true_coll == 0)).sum())
        fn = int(((A_rec == 0) & (A_true_coll > 0)).sum())
        row_str += f'  TP={tp} FP={fp} FN={fn}'
    print(row_str)

print()
print('TP = true edges recovered  |  FP = spurious edges added  |  FN = true edges missed')


In [ ]:
# Visual inspection: show recovered CPDAGs for extreme combinations
configs = [
    (20,   1e-8, 'n=20,  α=0.001'),
    (50,   0.20,  'n=50,  α=0.20'),
    (2000, 0.001, 'n=2000, α=0.001'),
    (2000, 0.05,  'n=2000, α=0.05'),
]

adj_list, titles = [A_coll], ['True Collider']
for n, a, title in configs:
    np.random.seed(42)
    df = scm_coll.sample(n_samples=n)
    cg = pc(df[col_names].values, alpha=a, indep_test=fisherz)
    adj_list.append(get_adjacency_pc(cg, col_names).values)
    titles.append(title)

plot_graphs_from_adj(adj_list, Y_idx_list=[d] * len(adj_list), plot_titles=titles)


### 1.5.8 Step 4 — GES: score-based discovery and scoring strategies

Unlike PC, **GES (Greedy Equivalence Search)** does not perform independence tests. Instead it:
1. Defines a **score** over equivalence classes of DAGs (typically BIC or BDeu).
2. **Forward phase**: greedily inserts edges that most improve the score.
3. **Backward phase**: removes edges whose removal improves the score.

This makes GES:
- **Independent of α** — no threshold to tune.
- Potentially more robust to small samples when the score is well-calibrated.
- But sensitive to the **choice of score function**.

`causal-learn` supports two score functions for continuous data:
- `local_score_BIC` — Bayesian Information Criterion (linear Gaussian assumption).
- `local_score_CV_general` — cross-validation based score, model-agnostic but slower.

We compare both on our three SCMs across different sample sizes.


---
## Section 1.6 — GES: a score-based alternative to PC

PC works by running statistical tests and removing edges when independence holds.
That makes it sensitive to one thing: the choice of **α**.
Too low and spurious edges survive; too high and real edges disappear.

**GES (Greedy Equivalence Search)** takes a completely different route.
Instead of testing, it *scores* graphs — how well does this graph explain the data?
Then it searches for the highest-scoring equivalence class, in two greedy passes.

> **No α to tune.** GES replaces a threshold with a score function,
> which is typically better calibrated and more stable at small sample sizes.


### How GES works — the two phases

GES always stays within the space of **CPDAGs** (equivalence classes), never committing to a single DAG.

**Phase 1 — Forward (insert edges)**  
Start from the empty graph. At each step, add whichever edge raises the score the most. Stop when no single insertion helps.

**Phase 2 — Backward (remove edges)**  
Now try deleting edges one by one. Accept removals that improve the score. Stop when no single deletion helps.

The result is a CPDAG — the same kind of output as PC.

**What score?**  
The default is **BIC** (Bayesian Information Criterion):
it rewards fit to the data and penalises the number of edges — a built-in Occam's razor.
For linear Gaussian data, BIC is well-suited and fast.


### Step 1 — Import GES and run it on the three SCMs

The `causal-learn` API is straightforward:
```python
Record = ges(X, score_func='local_score_BIC')
Record['G'].graph  # adjacency matrix of the recovered CPDAG
```
No α, no independence test — just the data matrix and a score function.


In [ ]:
from causallearn.search.ScoreBased.GES import ges

ges_ch = ges(data_chain, 'local_score_BIC')
ges_fo = ges(data_fork,  'local_score_BIC')
ges_co = ges(data_coll,  'local_score_BIC')

A_ges_ch = get_adjacency_ges(ges_ch, col_names)
A_ges_fo = get_adjacency_ges(ges_fo, col_names)
A_ges_co = get_adjacency_ges(ges_co, col_names)

### Step 2 — Compare GES output to the true graphs

Let's plot GES side by side with the true DAGs — just as we did for PC.
We expect GES to recover the same CPDAGs:
- Chain and Fork should look identical to each other (same MEC — undirected X0 — X1 — Y).
- Collider should be the only one with both arrows oriented (v-structure is identifiable).


In [ ]:
plot_graphs_from_adj(
    [A_chain, A_ges_ch.values],
    Y_idx_list=[d, d],
    plot_titles=['True: Chain', 'GES recovered'])

plot_graphs_from_adj(
    [A_fork, A_ges_fo.values],
    Y_idx_list=[d, d],
    plot_titles=['True: Fork', 'GES recovered'])

plot_graphs_from_adj(
    [A_coll, A_ges_co.values],
    Y_idx_list=[d, d],
    plot_titles=['True: Collider', 'GES recovered'])


### Step 3 — Does GES handle small samples better than PC?

PC's biggest weakness is at low n: statistical tests lose power and the recovered skeleton becomes unreliable.
GES uses a score instead of a threshold, so it degrades more gracefully — at least in theory.

Let's run the same grid experiment as before (varying n and comparing results on the Collider SCM),
but now comparing PC and GES head to head.


In [ ]:
sample_sizes = [50, 200, 500, 2000]

print(f'{"":22} PC (α=0.05)          GES (BIC)')
print(f'{"":22} TP  FP  FN           TP  FP  FN')
print('-' * 65)

for n in sample_sizes:
    np.random.seed(42)
    df = scm_coll.sample(n_samples=n)
    X_np = df[col_names].values

    # PC
    cg_pc  = pc(X_np, alpha=0.05, indep_test=fisherz)
    A_pc   = get_adjacency_pc(cg_pc, col_names).values

    # GES
    rec    = ges(X_np, score_func='local_score_BIC')
    A_ges  = get_adjacency_ges(rec, col_names).values

    def edge_counts(A_rec, A_true):
        tp = int(((A_rec > 0) & (A_true > 0)).sum())
        fp = int(((A_rec > 0) & (A_true == 0)).sum())
        fn = int(((A_rec == 0) & (A_true > 0)).sum())
        return tp, fp, fn

    tp_pc, fp_pc, fn_pc   = edge_counts(A_pc,  A_coll)
    tp_ges, fp_ges, fn_ges = edge_counts(A_ges, A_coll)

    print(f'n={n:<6}  (collider)   '
          f'TP={tp_pc} FP={fp_pc} FN={fn_pc}        '
          f'TP={tp_ges} FP={fp_ges} FN={fn_ges}')

print()
print('TP = true edges recovered  |  FP = spurious edges  |  FN = missed edges')


### Step 4 — Visualise the difference at low n

Numbers tell us what happened, but graphs show us *how* each algorithm failed.
At n=50, does GES add spurious edges or miss real ones?
Does it at least preserve the v-structure that PC struggles to identify?


In [ ]:
configs = [
    (20,   'n=20'),
    (200,  'n=200'),
    (500,  'n=500')
]

adj_list = [A_coll]
titles   = ['True Collider']

for n, label in configs:
    np.random.seed(42)
    df = scm_coll.sample(n_samples=n)
    X_np = df[col_names].values

    # GES
    rec = ges(X_np, score_func='local_score_BIC')
    adj_list.append(get_adjacency_ges(rec, col_names).values)
    titles.append(f'GES  {label}')

plot_graphs_from_adj(adj_list, Y_idx_list=[d] * len(adj_list), plot_titles=titles)


### What GES can and cannot do

GES, like PC, outputs a **CPDAG** — the Markov equivalence class of the true DAG.
This means:

- Chain and Fork will always look **identical** to GES (and to PC). No score-based method can distinguish them — they encode exactly the same conditional independencies.
- The Collider is the exception: the v-structure `X0 → X1 ← Y` is uniquely identifiable, and GES will orient both arrows correctly given enough data.

**The fundamental limit** is the MEC itself — not the algorithm.
To go beyond it and orient edges within an equivalence class,
we need a different kind of assumption entirely.
That's what LinGAM does.
